# Italian Electoral Data — PySpark Lakehouse Pipeline

Full **Bronze → Silver → Gold** pipeline over Italian electoral CSV data  
(Camera and Comunali elections, 1948–2025, 243 files, 57 distinct source schemas).

## Lakehouse layers

| Layer | Path | What it contains |
|---|---|---|
| Bronze | `data_parquet/bronze/` | All CSVs read as strings, columns renamed to English, metadata attached |
| Silver | `data_parquet/silver/` | Unified typed schema, surrogate keys pre-computed |
| Gold | `data_parquet/gold/` | Star schema: 4 dimension tables + 3 fact tables |

## Star schema

```
         dim_election          dim_geography
               │                     │
               └──────────┬──────────┘
                           │
              ┌────────────┼────────────┐
              │            │            │
       fact_turnout  fact_list    fact_candidate
                     _votes        _votes
                          │            │
                     dim_list    dim_candidate
```

All fact tables are partitioned by `(election_type, year)` for partition pruning at scale.

## 0. Setup

In [39]:
import os
import re
from functools import reduce

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("ItalianElectoralDataPipeline")
    # Adaptive Query Execution: automatically coalesces shuffle partitions
    # and switches join strategies at runtime — critical for skewed data.
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready")

Spark 3.5.3 ready


In [40]:
# ── Paths ──────────────────────────────────────────────────────────────────
PROJECT_ROOT = "/Users/hajdumarcell/Documents/GitHub/BDA_project"
VIZ_ROOT     = os.path.join(PROJECT_ROOT, "Visualization")
DATA_CSV     = os.path.join(VIZ_ROOT, "data_csv")
BRONZE_PATH  = os.path.join(VIZ_ROOT, "data_parquet", "bronze")
SILVER_PATH  = os.path.join(VIZ_ROOT, "data_parquet", "silver")
GOLD_PATH    = os.path.join(VIZ_ROOT, "data_parquet", "gold")

for p in [BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
    os.makedirs(p, exist_ok=True)

print("Output directories ready")
print(f"DATA_CSV: {DATA_CSV}")

Output directories ready
DATA_CSV: /Users/hajdumarcell/Documents/GitHub/BDA_project/Visualization/data_csv


## 1. Bronze Layer — Raw Ingestion

### 1.1 File Inventory

Walk the directory tree and collect metadata for every CSV file we intend to ingest.  
Skipped:
- `links/` folder (download manifests, not election data)
- `Legenda.csv` files (schema documentation files, contain only a `prova` column)

In [41]:
SKIP_PATTERNS = ["legenda", "link"]

# The 2017-11-05 Comunali file is missing its header row entirely.
# Column order was reconstructed by matching data values against the 2016 schema.
CORRUPTED_FILE = "comunali-20171105.csv"
CORRUPTED_COLS = [
    "region", "province", "municipality",
    "registered_voters", "registered_voters_male", "blank_ballots",
    "list_name", "list_votes", "seats_won",
    "surname", "given_name", "birth_date_raw", "birth_place", "sex",
    "elected_flag", "candidate_votes",
]


def collect_csv_files(root_dir: str) -> list:
    """
    Return a list of dicts describing every CSV file to ingest.
    Each dict contains: path, election_type, folder_date, filename.
    """
    records = []
    for dirpath, _, filenames in os.walk(root_dir):
        rel = os.path.relpath(dirpath, root_dir)
        if rel.startswith("links"):
            continue
        for fname in filenames:
            if not fname.lower().endswith(".csv"):
                continue
            if any(p in fname.lower() for p in SKIP_PATTERNS):
                continue
            parts = rel.replace("\\", "/").split("/")
            if   parts[0].lower() == "camera":   election_type = "CHAMBER"
            elif parts[0].lower() == "comunali": election_type = "MUNICIPAL"
            else:                                 election_type = "UNKNOWN"
            folder_date = (
                parts[1]
                if len(parts) > 1 and re.match(r"^\d{8}$", parts[1])
                else None
            )
            records.append({
                "path":          os.path.join(dirpath, fname),
                "election_type": election_type,
                "folder_date":   folder_date,
                "filename":      fname,
            })
    return records


inventory = collect_csv_files(DATA_CSV)
print(f"Files to ingest: {len(inventory)}")

# Preview a sample
for r in sorted(inventory, key=lambda x: (x["election_type"], x["folder_date"] or ""))[:8]:
    print(f"  {r['election_type']:10s}  {r['folder_date']}  {r['filename']}")

Files to ingest: 239
  CHAMBER     19480418  camera-19480418.csv
  CHAMBER     19480418  camera1948_preferenze.csv
  CHAMBER     19530607  camera-19530607.csv
  CHAMBER     19530607  camera1953_preferenze.csv
  CHAMBER     19580525  camera1958_preferenze.csv
  CHAMBER     19580525  camera-19580525.csv
  CHAMBER     19630428  camera1963_preferenze.csv
  CHAMBER     19630428  camera-19630428.csv


### 1.2 Column Rename Map

All 57 source schemas use varying Italian column names (mixed case, abbreviations,
typos). This map normalises every known variant to a single English name.
Unknown columns are left as-is and will end up as `null`-filled columns in the
unified DataFrame.

> **Note on `TOTVOTI`:** in Estero list files this column represents list votes;
> in Camera uninominal candidate files (1994–2001) it represents the candidate's
> personal vote total. We map it to `list_votes` here and fix the uninominal case
> in Silver.

In [42]:
RENAME_MAP = {
    # ── Election metadata ───────────────────────────────────────────────────
    "DATAELEZIONE": "election_date_raw",  "DataElezione": "election_date_raw",
    "CODTIPOELEZIONE": "election_type_code",
    "TURNO": "round",

    # ── Domestic geography ──────────────────────────────────────────────────
    "REGIONE": "region",    "regione": "region",    "Regione": "region",
    "PROVINCIA": "province", "provincia": "province", "Provincia": "province",
    "COMUNE": "municipality", "comune": "municipality", "Comune": "municipality",
    "ComuneSut": "municipality",  # 1980 typo in source data

    # ── Electoral geography ─────────────────────────────────────────────────
    "CIRCOSCRIZIONE": "electoral_district",
    "CIRCOCRIZIONE":  "electoral_district",  # 2018 typo in source data
    "circ": "electoral_district",
    "COLLEGIO": "constituency",
    "COLLEGIOPLURINOMINALE": "constituency",
    "COLLPLURINOM":          "constituency",
    "COLLEGIOPLURINOM":      "constituency",
    "coll": "constituency",
    "COLLEGIOUNINOM":       "uninominal_constituency",
    "COLLEGIOUNINOMINALE":  "uninominal_constituency",

    # ── Abroad geography ────────────────────────────────────────────────────
    "RIPARTIZIONE": "abroad_zone",
    "NAZIONE":      "country",

    # ── Registered voters ───────────────────────────────────────────────────
    "ELETTORI": "registered_voters",
    "ELETTORI_TOTALI": "registered_voters",  "ELETTORITOTALI": "registered_voters",
    "ELETTORITOT": "registered_voters",       "elettoritot": "registered_voters",
    "ElettoriTotali": "registered_voters",    "NUMELETTORITOTALI": "registered_voters",
    "ELETTORI_MASCHI": "registered_voters_male",
    "ELETTORIMASCHI": "registered_voters_male", "elettorimaschi": "registered_voters_male",
    "NUMELETTORIMASCHI": "registered_voters_male", "ELETTORIM": "registered_voters_male",

    # ── Turnout ─────────────────────────────────────────────────────────────
    "VOTANTI": "turnout_total",
    "VOTANTI_TOTALI": "turnout_total",  "VOTANTITOTALI": "turnout_total",
    "VOTANTITOT": "turnout_total",      "votantitot": "turnout_total",
    "NUMVOTANTITOTALI": "turnout_total", "VotantiTotali": "turnout_total",
    "VOTANTI_MASCHI": "turnout_male",
    "VOTANTIMASCHI": "turnout_male",  "votantimaschi": "turnout_male",
    "NUMVOTANTIMASCHI": "turnout_male", "VOTANTIM": "turnout_male",

    # ── Ballot counts ───────────────────────────────────────────────────────
    "SCHEDE_BIANCHE": "blank_ballots",  "SCHEDEBIANCHE": "blank_ballots",
    "SKBIANCHE": "blank_ballots",       "skbianche": "blank_ballots",
    "SchedeBianche": "blank_ballots",
    "SCHEDECONTESTATE": "contested_ballots",
    "SCHEDENONVALIDE": "invalid_ballots",  "SKNONVALIDE": "invalid_ballots",

    # ── Aggregate valid votes ────────────────────────────────────────────────
    "VOTIVALIDI": "valid_votes",
    "VOTIVALIDILISTE": "valid_list_votes",   "votivalidiliste": "valid_list_votes",
    "VOTIVALIDICAND": "valid_candidate_votes",
    "votivalidicandidsindaco": "valid_candidate_votes",

    # ── Hourly turnout (2022 Comunali "Votanti Varie Ore" files only) ────────
    "VOT1": "turnout_hour_1", "VOT2": "turnout_hour_2", "VOT3": "turnout_hour_3",

    # ── Party / list ────────────────────────────────────────────────────────
    "LISTA": "list_name",  "descrlista": "list_name",  "DESCRLISTA": "list_name",
    "descrcontrass": "list_name",  # uninominal opponent coalition label
    "VOTI_LISTA": "list_votes",  "VOTILISTA": "list_votes",
    "votiLista": "list_votes",   "VOTI": "list_votes",   "VotiLista": "list_votes",
    "TOTVOTI": "list_votes",     # see docstring above
    "SEGGI_LISTA": "seats_won",  "SEGGILISTA": "seats_won",
    "seggilista": "seats_won",   "SeggiLista": "seats_won",  "SEGGI": "seats_won",
    "NUMLISTA": "list_number",

    # ── Candidate personal info ─────────────────────────────────────────────
    "COGNOME": "surname",   "cognome": "surname",
    "NOME": "given_name",   "nome": "given_name",
    "DATANASCITA": "birth_date_raw",  "datanascita": "birth_date_raw",
    "DATA_NASCITA": "birth_date_raw",
    "LUOGONASCITA": "birth_place",    "luogonascita": "birth_place",
    "LUOGO_NASCITA": "birth_place",
    "SESSO": "sex",  "sesso": "sex",

    # ── Candidate vote counts ───────────────────────────────────────────────
    "VOTI_CANDIDATO":   "candidate_votes",
    "VOTICANDIDSINDACO": "candidate_votes",   # 2021 Comunali
    "VOTICAND":          "candidate_votes",
    "VOTICANDLEADER":    "candidate_votes",   # 2022-11 Comunali

    # ── Elected status ──────────────────────────────────────────────────────
    "ELETTO": "elected_flag",             # old files: S / empty
    "CODTIPOELETTO": "elected_type_code", # new files: S=Mayor C=Councillor N=Not elected
    "codtipoeletto": "elected_type_code",

    # ── Camera-specific ─────────────────────────────────────────────────────
    "PREFERENZE": "preference_votes",

    # ── Other ───────────────────────────────────────────────────────────────
    "NUMCANDIDATO": "candidate_number",
    "CODTIPOINVALIDITA": "invalid_type_code",
    "codtipoinvalidita": "invalid_type_code",
}

print(f"Rename map entries: {len(RENAME_MAP)}")

Rename map entries: 105


### 1.3 Ingest All CSV Files

Every file is read with `inferSchema=False` (all strings) to avoid Spark making wrong
type assumptions on messy data. The rename map is applied per-file **before** the
`unionByName` so that variant spellings of the same concept (e.g. `VOTI_LISTA` vs
`VOTILISTA`) are merged into one column rather than proliferating as separate nulls.

In [43]:
def _apply_rename(df: DataFrame) -> DataFrame:
    """Rename columns present in df according to RENAME_MAP; unknown columns kept as-is."""
    for src, tgt in RENAME_MAP.items():
        if src in df.columns:
            df = df.withColumnRenamed(src, tgt)
    return df


def read_file(info: dict) -> DataFrame:
    """
    Read one CSV file, apply the rename map, and attach three metadata columns:
      _source_file    — full path, kept through Silver for edge-case logic in Gold
      _election_type  — CHAMBER / MUNICIPAL
      _folder_date    — yyyymmdd string from the parent folder name
    All data values remain as strings at this stage.
    """
    path, fname = info["path"], info["filename"]

    if fname == CORRUPTED_FILE:
        # Missing header: read positionally and assign reconstructed column names.
        df = (
            spark.read
            .option("header", "false")
            .option("encoding", "UTF-8")
            .csv(path)
        )
        for i, col_name in enumerate(CORRUPTED_COLS):
            df = df.withColumnRenamed(f"_c{i}", col_name)
    elif fname.startswith("_"):
        # Hadoop/Spark treats files whose names start with "_" as hidden and silently
        # drops them, producing an empty dataset and an UNABLE_TO_INFER_SCHEMA error.
        # Bypass by reading via pandas (Python file I/O, no Hadoop filter).
        import pandas as pd
        pdf = pd.read_csv(path, dtype=str, encoding="utf-8", keep_default_na=False)
        df = spark.createDataFrame(pdf)
        df = _apply_rename(df)
    else:
        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "false")
            .option("encoding", "UTF-8")
            .csv(path)
        )
        df = _apply_rename(df)

    return (
        df
        .withColumn("_source_file",   F.lit(path))
        .withColumn("_election_type", F.lit(info["election_type"]))
        .withColumn("_folder_date",   F.lit(info["folder_date"]))
    )


# Read all files; allowMissingColumns=True fills absent columns with null.
# This is the standard Spark pattern for schema-on-read over heterogeneous sources.
dfs = [read_file(info) for info in inventory]
bronze = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

print(f"Bronze: {bronze.count():,} rows across {len(bronze.columns)} columns")

Bronze: 2,510,128 rows across 49 columns


### 1.4 Persist Bronze to Parquet

Partitioned by `_election_type` (2 values). Subsequent layers load from this Parquet
so the expensive CSV parse runs only once.

In [44]:
(
    bronze
    .write
    .mode("overwrite")
    .partitionBy("_election_type")
    .parquet(BRONZE_PATH)
)
print(f"Bronze written → {BRONZE_PATH}")

26/05/31 13:36:27 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB


Bronze written → /Users/hajdumarcell/Documents/GitHub/BDA_project/Visualization/data_parquet/bronze


## 2. Silver Layer — Schema Normalization

Silver transforms Bronze from a bag of strings into a properly typed, deduplicated,
key-enriched DataFrame. Each transformation is a pure function so they can be tested
and composed independently.

### 2.1 Numeric Cleaning

Camera uninominal files (1994–2001) store numbers in Italian decimal format:
`"5181,00"` means 5181. We strip the fractional part before casting.

In [45]:
NUMERIC_COLS = [
    "registered_voters", "registered_voters_male",
    "turnout_total",     "turnout_male",
    "blank_ballots",     "contested_ballots",  "invalid_ballots",
    "valid_votes",       "valid_list_votes",   "valid_candidate_votes",
    "list_votes",        "seats_won",           "list_number",
    "candidate_votes",   "preference_votes",
    "turnout_hour_1",    "turnout_hour_2",      "turnout_hour_3",
    "round",
]


def clean_numerics(df: DataFrame) -> DataFrame:
    """
    Strip Italian decimal commas then cast to long.
    Pattern r",\d*$" removes a trailing comma and any digits after it.
    Examples: "5181,00" → 5181   |   "12345" → 12345   |   null → null
    """
    for col_name in NUMERIC_COLS:
        if col_name in df.columns:
            df = df.withColumn(
                col_name,
                F.regexp_replace(F.col(col_name), r",\d*$", "").cast("long"),
            )
    return df

<>:16: SyntaxWarning: invalid escape sequence '\d'
<>:16: SyntaxWarning: invalid escape sequence '\d'
/var/folders/td/4zhft30j4nscpqzk9nkxmxmh0000gn/T/ipykernel_22046/2146211814.py:16: SyntaxWarning: invalid escape sequence '\d'
  Pattern r",\d*$" removes a trailing comma and any digits after it.


### 2.2 Date Parsing

Two date fields are present:

| Field | Source | Strategy |
|---|---|---|
| `election_date` | `_folder_date` column (always `yyyyMMdd`) | Unambiguous — always use this |
| `birth_date` | `birth_date_raw` column (multiple formats) | Try formats in order, take first match |

In [46]:
BIRTH_DATE_FORMATS = [
    "d/M/yyyy HH:mm:ss",   # 16/5/1957 00:00:00
    "dd/MM/yyyy HH:mm:ss", # 23/11/1948 00:00:00
    "d/M/yyyy",            # 19/3/1984
    "dd/MM/yyyy",          # 19/03/1954
    "yyyy-MM-dd",          # 1953-11-29
]


def parse_dates(df: DataFrame) -> DataFrame:
    """
    Derive election_date from the folder name (authoritative, unambiguous).
    Parse birth_date from birth_date_raw using a chain of format attempts.
    Drop raw string columns once parsing is complete.
    """
    df = df.withColumn(
        "election_date",
        F.to_date(F.col("_folder_date"), "yyyyMMdd"),
    )
    if "birth_date_raw" in df.columns:
        df = df.withColumn(
            "birth_date",
            F.coalesce(*[
                F.to_date(F.col("birth_date_raw"), fmt)
                for fmt in BIRTH_DATE_FORMATS
            ]),
        ).drop("birth_date_raw")
    if "election_date_raw" in df.columns:
        df = df.drop("election_date_raw")
    return df

### 2.3 Elected Status Normalisation

Two different conventions exist across eras:

| Era | Column | Values |
|---|---|---|
| Pre-2003 | `elected_flag` (from `ELETTO`) | `S` = elected, blank = not |
| 2003+ | `elected_type_code` (from `CODTIPOELETTO`) | `S` = Mayor, `C` = Councillor, `N` = not elected |

In [47]:
def normalize_elected(df: DataFrame) -> DataFrame:
    """
    Derive a boolean `elected` column from whichever source column is present.
    `elected_type_code` is retained as-is for downstream analysis.
    """
    if "elected_flag" in df.columns:
        df = df.withColumn(
            "elected",
            F.when(F.upper(F.col("elected_flag")) == "S", True).otherwise(False),
        ).drop("elected_flag")

    if "elected_type_code" in df.columns and "elected" not in df.columns:
        df = df.withColumn(
            "elected",
            F.when(F.upper(F.col("elected_type_code")).isin("S", "C"), True)
             .otherwise(False),
        )
    return df

### 2.4 Duplicate Column Guard

If the same file happens to contain two source column names that both map to
the same English target, `withColumnRenamed` creates two identically named columns.
This function detects and drops the second occurrence.

In [48]:
def deduplicate_columns(df: DataFrame) -> DataFrame:
    """Keep only the first occurrence of each column name."""
    seen, suffixed = set(), []
    for name in df.columns:
        if name in seen:
            suffixed.append(f"__drop_{name}_{len(seen)}")
        else:
            seen.add(name)
            suffixed.append(name)
    df = df.toDF(*suffixed)
    drop_cols = [c for c in df.columns if c.startswith("__drop_")]
    return df.drop(*drop_cols) if drop_cols else df

### 2.5 Surrogate Key Helper

In [49]:
def _make_id(*col_names) -> F.Column:
    """
    MD5 hash of pipe-delimited column values cast to string.
    Nulls are coerced to empty string so the key is never null.
    MD5 is sufficient here — collision probability is negligible at this
    data scale and the key only needs to be stable, not cryptographically secure.
    """
    parts = [F.coalesce(F.col(c).cast("string"), F.lit("")) for c in col_names]
    return F.md5(F.concat_ws("||", *parts))

### 2.6 Assemble Silver

All transformations are applied in sequence. Four surrogate keys are computed here
so both the dimension tables and fact tables can be built from Silver without
requiring additional joins.

In [50]:
# Load from persisted Bronze (safe to re-run this cell independently)
bronze = spark.read.parquet(BRONZE_PATH)

silver = (
    bronze
    .transform(deduplicate_columns)
    .transform(clean_numerics)
    .transform(parse_dates)
    .transform(normalize_elected)
    # Promote private metadata columns to first-class citizens
    .withColumnRenamed("_election_type", "election_type")
    .withColumnRenamed("_source_file",   "source_file")
    # Year derived from election_date — used for Parquet partitioning
    .withColumn("year", F.year("election_date"))
    # Drop helper columns that have served their purpose
    .drop("_folder_date", "election_type_code", "candidate_number")
)

# ── Surrogate keys ──────────────────────────────────────────────────────────
# election_id: one key per (date × type) pair
silver = silver.withColumn(
    "election_id", _make_id("election_date", "election_type")
)

# geo_id: one key per unique combination of geographic identifiers.
# Using all fields means municipality-level and district-level records
# get distinct keys even when they share a region/province.
silver = silver.withColumn(
    "geo_id",
    _make_id("region", "province", "municipality",
             "electoral_district", "constituency", "uninominal_constituency",
             "abroad_zone", "country"),
)

# list_id: scoped to election so "Lega 2018" ≠ "Lega 2022"
silver = silver.withColumn(
    "list_id",
    F.when(
        F.col("list_name").isNotNull(),
        _make_id("election_id", "list_name"),
    ),
)

# candidate_id: identity is (surname, given_name, birth_date).
# Candidates missing birth_date get a key based on name only — acceptable
# for older data where birth dates were not recorded.
silver = silver.withColumn(
    "candidate_id",
    F.when(
        F.col("surname").isNotNull(),
        _make_id("surname", "given_name", "birth_date"),
    ),
)

# ── Fix uninominal candidate files (Camera 1994–2001) ───────────────────────
# In these files TOTVOTI (the candidate's personal vote total in their
# uninominal constituency) was mapped to list_votes by the rename map.
# We copy it to candidate_votes and will exclude these rows from fact_list_votes.
silver = silver.withColumn(
    "candidate_votes",
    F.when(
        F.col("candidate_votes").isNull()
        & F.col("list_votes").isNotNull()
        & F.col("source_file").rlike(r"[Uu]ninom_[Cc]and"),
        F.col("list_votes"),
    ).otherwise(F.col("candidate_votes")),
)

silver.write.mode("overwrite").partitionBy("election_type", "year").parquet(SILVER_PATH)
print(f"Silver written → {SILVER_PATH}")
print(f"Columns: {len(silver.columns)}")
silver.printSchema()

Silver written → /Users/hajdumarcell/Documents/GitHub/BDA_project/Visualization/data_parquet/silver
Columns: 51
root
 |-- electoral_district: string (nullable = true)
 |-- constituency: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- registered_voters: long (nullable = true)
 |-- registered_voters_male: long (nullable = true)
 |-- turnout_total: long (nullable = true)
 |-- turnout_male: long (nullable = true)
 |-- valid_votes: long (nullable = true)
 |-- blank_ballots: long (nullable = true)
 |-- source_file: string (nullable = true)
 |-- list_name: string (nullable = true)
 |-- list_votes: long (nullable = true)
 |-- surname: string (nullable = true)
 |-- given_name: string (nullable = true)
 |-- birth_place: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- elected_type_code: string (nullable = true)
 |-- abroad_zone: string (nullable = true)
 |-- country: string (nullable = true)
 |-- invalid_ballots: long (nullable = true)
 |-- province: str

## 3. Gold Layer — Star Schema

All tables are built from Silver by projecting and deduplicating.  
No joins are needed at this stage because every surrogate key was pre-computed in Silver.

Fact tables are partitioned by `(election_type, year)` — a query like
`WHERE election_type='MUNICIPAL' AND year=2019` reads only one partition file.

### 3.1 `dim_election`

In [51]:
silver = spark.read.parquet(SILVER_PATH)  # load from disk (re-run safe)

dim_election = (
    silver
    .select("election_id", "election_date", "election_type", "year")
    .distinct()
    .orderBy("election_date")
)

dim_election.write.mode("overwrite").parquet(os.path.join(GOLD_PATH, "dim_election"))
print(f"dim_election: {dim_election.count()} rows")
dim_election.show(10)

dim_election: 178 rows
+--------------------+-------------+-------------+----+
|         election_id|election_date|election_type|year|
+--------------------+-------------+-------------+----+
|785adc15ffb92afde...|   1948-04-18|      CHAMBER|1948|
|f9382923b46802db6...|   1953-06-07|      CHAMBER|1953|
|27ff780aa734d33f3...|   1958-05-25|      CHAMBER|1958|
|360b8c9bd0ab6491d...|   1963-04-28|      CHAMBER|1963|
|1247d782e29cfe7c3...|   1968-05-19|      CHAMBER|1968|
|09682dcaa4b37069e...|   1970-06-07|    MUNICIPAL|1970|
|379462f2cf63e6db9...|   1972-05-07|      CHAMBER|1972|
|1a14571f1e63a6f49...|   1975-06-15|    MUNICIPAL|1975|
|8705dbe794e5e6d6d...|   1976-06-20|      CHAMBER|1976|
|2ac8703ff5a208a28...|   1979-06-03|      CHAMBER|1979|
+--------------------+-------------+-------------+----+
only showing top 10 rows



### 3.2 `dim_geography`

`geo_scope` classifies each record so queries can filter to the right level
without inspecting which columns are populated.

In [52]:
GEO_COLS = [
    "geo_id", "region", "province", "municipality",
    "electoral_district", "constituency", "uninominal_constituency",
    "abroad_zone", "country",
]

dim_geography = (
    silver
    .select(*[c for c in GEO_COLS if c in silver.columns])
    .distinct()
    .withColumn(
        "geo_scope",
        F.when(
            F.col("abroad_zone").isNotNull() | F.col("country").isNotNull(),
            "COUNTRY",
        ).when(
            F.col("uninominal_constituency").isNotNull(),
            "CONSTITUENCY",
        ).when(
            F.col("municipality").isNotNull(),
            "MUNICIPALITY",
        ).otherwise("DISTRICT"),
    )
)

dim_geography.write.mode("overwrite").parquet(os.path.join(GOLD_PATH, "dim_geography"))
print(f"dim_geography: {dim_geography.count():,} rows")
dim_geography.groupBy("geo_scope").count().orderBy("count", ascending=False).show()

dim_geography: 54,256 rows


+------------+-----+
|   geo_scope|count|
+------------+-----+
|MUNICIPALITY|45744|
|CONSTITUENCY| 8144|
|     COUNTRY|  203|
|    DISTRICT|  165|
+------------+-----+



### 3.3 `dim_list`

Lists are scoped to their election — the same party name across different elections
produces different `list_id` values. This is intentional: list vote totals are
election-specific and cross-election aggregation should be done at query time
by matching on `list_name`.

In [53]:
dim_list = (
    silver
    .filter(F.col("list_name").isNotNull())
    .select("list_id", "election_id", "list_name", "election_type")
    .distinct()
)

dim_list.write.mode("overwrite").parquet(os.path.join(GOLD_PATH, "dim_list"))
print(f"dim_list: {dim_list.count():,} rows")
dim_list.show(5)

dim_list: 61,363 rows
+--------------------+--------------------+--------------------+-------------+
|             list_id|         election_id|           list_name|election_type|
+--------------------+--------------------+--------------------+-------------+
|68c5390cdcc9f6ccc...|03f7702d6671a837b...|RIFONDAZIONE COMU...|      CHAMBER|
|8ae570ea9b7c788c2...|02d9367b569fe6bbe...|                 310|      CHAMBER|
|d8a88ebea9d82b09f...|02d9367b569fe6bbe...|                 220|      CHAMBER|
|5de81d92517fd4208...|8fc552806b04f2668...|    U.D.EUR POPOLARI|    MUNICIPAL|
|1cdfe601f8fb061a3...|8fc552806b04f2668...|PUNTO A CAPO PER ...|    MUNICIPAL|
+--------------------+--------------------+--------------------+-------------+
only showing top 5 rows



### 3.4 `dim_candidate`

In [54]:
CANDIDATE_ATTRS = ["surname", "given_name", "birth_date", "birth_place", "sex"]

dim_candidate = (
    silver
    .filter(F.col("surname").isNotNull())
    .select(
        "candidate_id",
        *[c for c in CANDIDATE_ATTRS if c in silver.columns],
    )
    .distinct()
)

dim_candidate.write.mode("overwrite").parquet(os.path.join(GOLD_PATH, "dim_candidate"))
print(f"dim_candidate: {dim_candidate.count():,} rows")
dim_candidate.show(5)

dim_candidate: 185,710 rows
+--------------------+---------+-----------+----------+-------------+---+
|        candidate_id|  surname| given_name|birth_date|  birth_place|sex|
+--------------------+---------+-----------+----------+-------------+---+
|a4ee2c3d18a184e5d...|   ZUCCHI| ALESSANDRO|1976-02-04|   VOGHERA PV|  M|
|37e937ccf13020ab9...|DEL CARLO|   GIUSEPPE|1939-06-07|    CAPANNORI|  M|
|b19e3a1791bae06d6...|   FAVATA|    ANTONIO|1962-09-22|CALTANISSETTA|  M|
|d9fd930783f47015b...|    PIRAS|MARIO BRUNO|1955-12-14|        GAIRO|  M|
|2e5f8677fe068ea77...|  INCERTI|  ANTONELLA|1960-07-07|        PARMA|  F|
+--------------------+---------+-----------+----------+-------------+---+
only showing top 5 rows



### 3.5 `fact_turnout`

Many source files (1993–2023 Comunali) repeat the same turnout figures on every
candidate/list row for the same municipality. `dropDuplicates` on the natural key
`(election_id, geo_id, round)` collapses these repetitions to one row.

In [55]:
TURNOUT_COLS = [
    "election_id", "geo_id", "election_type", "year", "round",
    "registered_voters", "registered_voters_male",
    "turnout_total",     "turnout_male",
    "blank_ballots",     "contested_ballots",  "invalid_ballots",
    "valid_votes",       "valid_list_votes",   "valid_candidate_votes",
]

fact_turnout = (
    silver
    .filter(F.col("registered_voters").isNotNull())
    .select(*[c for c in TURNOUT_COLS if c in silver.columns])
    .dropDuplicates(["election_id", "geo_id", "round"])
)

(
    fact_turnout
    .write.mode("overwrite")
    .partitionBy("election_type", "year")
    .parquet(os.path.join(GOLD_PATH, "fact_turnout"))
)
print(f"fact_turnout: {fact_turnout.count():,} rows")

fact_turnout: 229,039 rows


### 3.6 `fact_list_votes`

Uninominal candidate files (Camera 1994–2001) are excluded here because their
`list_votes` column actually holds a candidate total, not a list total.

In [56]:
LIST_VOTE_COLS = [
    "election_id", "geo_id", "list_id", "election_type", "year", "round",
    "list_votes", "seats_won",
]

fact_list_votes = (
    silver
    .filter(
        F.col("list_name").isNotNull()
        & F.col("list_votes").isNotNull()
        # Exclude uninominal candidate files where list_votes = candidate's personal total
        & ~F.col("source_file").rlike(r"[Uu]ninom_[Cc]and")
    )
    .select(*[c for c in LIST_VOTE_COLS if c in silver.columns])
    .dropDuplicates(["election_id", "geo_id", "list_id", "round"])
)

(
    fact_list_votes
    .write.mode("overwrite")
    .partitionBy("election_type", "year")
    .parquet(os.path.join(GOLD_PATH, "fact_list_votes"))
)
print(f"fact_list_votes: {fact_list_votes.count():,} rows")

fact_list_votes: 2,110,147 rows


### 3.7 `fact_candidate_votes`

In [57]:
CAND_VOTE_COLS = [
    "election_id", "geo_id", "candidate_id", "list_id",
    "election_type", "year", "round",
    "candidate_votes", "elected", "elected_type_code", "preference_votes",
]

fact_candidate_votes = (
    silver
    .filter(F.col("surname").isNotNull())
    .select(*[c for c in CAND_VOTE_COLS if c in silver.columns])
    .dropDuplicates(["election_id", "geo_id", "candidate_id", "list_id", "round"])
)

(
    fact_candidate_votes
    .write.mode("overwrite")
    .partitionBy("election_type", "year")
    .parquet(os.path.join(GOLD_PATH, "fact_candidate_votes"))
)
print(f"fact_candidate_votes: {fact_candidate_votes.count():,} rows")

fact_candidate_votes: 563,809 rows


## 4. Register Spark SQL Views

All Gold tables are registered as in-session temporary views so they can be
queried with plain SQL. Loading from Parquet makes this section independently
re-runnable — no need to re-run the pipeline cells above.

In [58]:
GOLD_TABLES = {
    "dim_election":         "dim_election",
    "dim_geography":        "dim_geography",
    "dim_list":             "dim_list",
    "dim_candidate":        "dim_candidate",
    "fact_turnout":         "fact_turnout",
    "fact_list_votes":      "fact_list_votes",
    "fact_candidate_votes": "fact_candidate_votes",
}

for view_name, folder in GOLD_TABLES.items():
    path = os.path.join(GOLD_PATH, folder)
    spark.read.parquet(path).createOrReplaceTempView(view_name)
    print(f"  registered: {view_name}")

print()
spark.sql("SHOW TABLES").show(truncate=False)

  registered: dim_election
  registered: dim_geography
  registered: dim_list
  registered: dim_candidate
  registered: fact_turnout
  registered: fact_list_votes
  registered: fact_candidate_votes

+---------+--------------------+-----------+
|namespace|tableName           |isTemporary|
+---------+--------------------+-----------+
|         |dim_candidate       |true       |
|         |dim_election        |true       |
|         |dim_geography       |true       |
|         |dim_list            |true       |
|         |fact_candidate_votes|true       |
|         |fact_list_votes     |true       |
|         |fact_turnout        |true       |
+---------+--------------------+-----------+



## 5. Example Analytical Queries

All queries use Spark SQL for readability. They can also be expressed with the
DataFrame API if preferred.

---

### Query 1 — National Turnout Rate by Year and Election Type

Long-run view of participation. Filters out abroad geography (`geo_scope = 'COUNTRY'`)
so only domestic votes are counted.

In [59]:
spark.sql("""
    SELECT
        e.year,
        e.election_type,
        COUNT(DISTINCT e.election_id)                                              AS elections_held,
        SUM(t.turnout_total)                                                       AS total_voters,
        SUM(t.registered_voters)                                                   AS total_registered,
        ROUND(100.0 * SUM(t.turnout_total) / NULLIF(SUM(t.registered_voters), 0), 2)
                                                                                   AS turnout_pct
    FROM fact_turnout   t
    JOIN dim_election   e USING (election_id)
    JOIN dim_geography  g USING (geo_id)
    WHERE t.registered_voters > 0
      AND g.geo_scope != 'COUNTRY'
    GROUP BY e.year, e.election_type
    ORDER BY e.year, e.election_type
""").show(40)

+----+-------------+--------------+------------+----------------+-----------+
|year|election_type|elections_held|total_voters|total_registered|turnout_pct|
+----+-------------+--------------+------------+----------------+-----------+
|1948|      CHAMBER|             1|    26774306|        29033012|      92.22|
|1953|      CHAMBER|             1|    28409140|        30276195|      93.83|
|1958|      CHAMBER|             1|    30434270|        32434429|      93.83|
|1963|      CHAMBER|             1|    31766009|        34199184|      92.89|
|1968|      CHAMBER|             1|    33001644|        35566493|      92.79|
|1970|    MUNICIPAL|             1|    26096389|        28419385|      91.83|
|1972|      CHAMBER|             1|    34525661|        37049353|      93.19|
|1975|    MUNICIPAL|             1|    27630326|        29942443|      92.28|
|1976|      CHAMBER|             1|    37743785|        40447569|      93.32|
|1979|      CHAMBER|             1|    38257912|        42213292

### Query 2 — Top 10 Lists by Votes in the 2022 Chamber Election

Aggregated at national level (all geo records summed). Shows both vote totals
and seat totals to reveal efficiency differences.

In [60]:
spark.sql("""
    SELECT
        l.list_name,
        SUM(lv.list_votes)  AS total_votes,
        SUM(lv.seats_won)   AS total_seats
    FROM fact_list_votes  lv
    JOIN dim_election     e  USING (election_id)
    JOIN dim_list         l  USING (list_id)
    WHERE e.election_date = '2022-09-25'
      AND e.election_type = 'CHAMBER'
    GROUP BY l.list_name
    ORDER BY total_votes DESC
    LIMIT 10
""").show(truncate=False)

+-----------------------------------------------------------+-----------+-----------+
|list_name                                                  |total_votes|total_seats|
+-----------------------------------------------------------+-----------+-----------+
|PARTITO DEMOCRATICO - ITALIA DEMOCRATICA E PROGRESSISTA    |305955     |NULL       |
|LEGA PER SALVINI PREMIER - FORZA ITALIA - FRATELLI D'ITALIA|282116     |NULL       |
|MOVIMENTO ASSOCIATIVO ITALIANI ALL'ESTERO - MAIE           |141356     |NULL       |
|MOVIMENTO 5 STELLE                                         |93371      |NULL       |
|UNIONE SUDAMERICANA EMIGRATI ITALIANI - USEI               |73241      |NULL       |
|AZIONE - ITALIA VIVA - CALENDA                             |60528      |NULL       |
|ALLEANZA VERDI E SINISTRA                                  |52994      |NULL       |
|+EUROPA                                                    |29971      |NULL       |
|MOVIMENTO DELLE LIBERTA'                             

### Query 3 — Seat Efficiency by Party (Municipal elections)

Votes-per-seat ratio shows how efficiently each party converts votes into
council seats. Higher = less efficient (votes wasted on losing races or
lists that exceeded proportional allocation).

In [61]:
spark.sql("""
    SELECT
        l.list_name,
        e.year,
        SUM(lv.list_votes)  AS total_votes,
        SUM(lv.seats_won)   AS total_seats,
        ROUND(
            SUM(lv.list_votes) / NULLIF(SUM(lv.seats_won), 0)
        )                   AS votes_per_seat
    FROM fact_list_votes  lv
    JOIN dim_election     e   USING (election_id)
    JOIN dim_list         l   USING (list_id)
    WHERE e.election_type = 'MUNICIPAL'
      AND lv.seats_won    > 0
    GROUP BY l.list_name, e.year
    HAVING SUM(lv.seats_won) >= 20         -- only parties that won at least 20 seats
    ORDER BY e.year DESC, total_seats DESC
    LIMIT 20
""").show(truncate=False)

+--------------------------------+----+-----------+-----------+--------------+
|list_name                       |year|total_votes|total_seats|votes_per_seat|
+--------------------------------+----+-----------+-----------+--------------+
|PARTITO DEMOCRATICO             |2025|171561     |151        |1136.0        |
|FRATELLI D'ITALIA               |2025|109841     |93         |1181.0        |
|FORZA ITALIA - PPE              |2025|48996      |35         |1400.0        |
|PARTITO DEMOCRATICO             |2023|293231     |371        |790.0         |
|FRATELLI D'ITALIA GIORGIA MELONI|2023|253829     |308        |824.0         |
|FORZA ITALIA                    |2023|83402      |90         |927.0         |
|PROGETTO COMUNE                 |2023|8929       |57         |157.0         |
|LEGA                            |2023|28316      |45         |629.0         |
|IMPEGNO COMUNE                  |2023|6701       |41         |163.0         |
|LEGA SALVINI PREMIER            |2023|28954      |3

### Query 4 — Abroad Participation Rate by Country

Italian citizens living abroad have voted in Chamber elections since 2006.
This query ranks countries by participation rate for the most recent election.

In [62]:
spark.sql("""
    SELECT
        g.country,
        g.abroad_zone,
        e.election_date,
        SUM(t.turnout_total)     AS voters_cast,
        SUM(t.registered_voters) AS eligible_voters,
        ROUND(
            100.0 * SUM(t.turnout_total) / NULLIF(SUM(t.registered_voters), 0), 2
        )                        AS participation_pct
    FROM fact_turnout   t
    JOIN dim_election   e USING (election_id)
    JOIN dim_geography  g USING (geo_id)
    WHERE g.geo_scope = 'COUNTRY'
      AND t.registered_voters > 0
    GROUP BY g.country, g.abroad_zone, e.election_date
    ORDER BY e.election_date DESC, participation_pct DESC
    LIMIT 30
""").show(30, truncate=False)

+-----------+---------------------------------+-------------+-----------+---------------+-----------------+
|country    |abroad_zone                      |election_date|voters_cast|eligible_voters|participation_pct|
+-----------+---------------------------------+-------------+-----------+---------------+-----------------+
|NEPAL      |AFRICA ASIA OCEANIA ANTARTIDE    |2022-09-25   |27         |34             |79.41            |
|SUDAN      |AFRICA ASIA OCEANIA ANTARTIDE    |2022-09-25   |94         |126            |74.60            |
|BURUNDI    |AFRICA ASIA OCEANIA ANTARTIDE    |2022-09-25   |37         |52             |71.15            |
|UZBEKISTAN |AFRICA ASIA OCEANIA ANTARTIDE    |2022-09-25   |26         |38             |68.42            |
|SRI LANKA  |AFRICA ASIA OCEANIA ANTARTIDE    |2022-09-25   |153        |243            |62.96            |
|BENIN      |AFRICA ASIA OCEANIA ANTARTIDE    |2022-09-25   |34         |54             |62.96            |
|SAN MARINO |EUROPA         

### Query 5 — Gender Representation Among Elected Officials

Tracks the share of female elected officials over time. Data coverage varies:
sex is recorded from the early 1990s onward for Comunali, and from 1994 for Camera.

In [63]:
spark.sql("""
    SELECT
        e.year,
        e.election_type,
        SUM(CASE WHEN c.sex = 'F' AND cv.elected THEN 1 ELSE 0 END) AS elected_female,
        SUM(CASE WHEN c.sex = 'M' AND cv.elected THEN 1 ELSE 0 END) AS elected_male,
        COUNT(CASE WHEN cv.elected THEN 1 END)                       AS total_elected,
        ROUND(
            100.0
            * SUM(CASE WHEN c.sex = 'F' AND cv.elected THEN 1 ELSE 0 END)
            / NULLIF(COUNT(CASE WHEN cv.elected THEN 1 END), 0),
            2
        )                                                            AS female_pct
    FROM fact_candidate_votes  cv
    JOIN dim_election           e  USING (election_id)
    JOIN dim_candidate          c  USING (candidate_id)
    WHERE c.sex IS NOT NULL
    GROUP BY e.year, e.election_type
    ORDER BY e.year, e.election_type
""").show(40)

+----+-------------+--------------+------------+-------------+----------+
|year|election_type|elected_female|elected_male|total_elected|female_pct|
+----+-------------+--------------+------------+-------------+----------+
|1948|      CHAMBER|             0|           0|            0|      NULL|
|1953|      CHAMBER|             0|           0|            0|      NULL|
|1958|      CHAMBER|             0|           0|            0|      NULL|
|1963|      CHAMBER|             0|           0|            0|      NULL|
|1968|      CHAMBER|             0|           0|            0|      NULL|
|1972|      CHAMBER|             0|           0|            0|      NULL|
|1976|      CHAMBER|             0|           0|            0|      NULL|
|1979|      CHAMBER|             0|           0|            0|      NULL|
|1983|      CHAMBER|             0|           0|            0|      NULL|
|1987|      CHAMBER|             0|           0|            0|      NULL|
|1992|      CHAMBER|             0|   

### Query 6 — Closest Municipal Elections (Smallest Winning Margin)

For each (election, municipality) pair, compute the vote gap between the
winner and the runner-up. The smallest margins reveal the most competitive races.

In [64]:
spark.sql("""
    WITH ranked AS (
        SELECT
            cv.election_id,
            cv.geo_id,
            cv.candidate_votes,
            ROW_NUMBER() OVER (
                PARTITION BY cv.election_id, cv.geo_id
                ORDER BY cv.candidate_votes DESC
            ) AS rank
        FROM fact_candidate_votes cv
        WHERE cv.candidate_votes IS NOT NULL
          AND cv.election_type   = 'MUNICIPAL'
    ),
    margins AS (
        SELECT
            election_id,
            geo_id,
            MAX(CASE WHEN rank = 1 THEN candidate_votes END) AS winner_votes,
            MAX(CASE WHEN rank = 2 THEN candidate_votes END) AS runner_up_votes
        FROM ranked
        GROUP BY election_id, geo_id
        HAVING MAX(CASE WHEN rank = 2 THEN candidate_votes END) IS NOT NULL
    )
    SELECT
        e.election_date,
        g.region,
        g.municipality,
        m.winner_votes,
        m.runner_up_votes,
        m.winner_votes - m.runner_up_votes AS margin
    FROM margins       m
    JOIN dim_election  e USING (election_id)
    JOIN dim_geography g USING (geo_id)
    ORDER BY margin ASC
    LIMIT 20
""").show(20, truncate=False)

+-------------+--------------+---------------------+------------+---------------+------+
|election_date|region        |municipality         |winner_votes|runner_up_votes|margin|
+-------------+--------------+---------------------+------------+---------------+------+
|1999-06-13   |LOMBARDIA     |PESCHIERA BORROMEO   |5639        |5639           |0     |
|1999-06-13   |EMILIA ROMAGNA|COPPARO              |8449        |8449           |0     |
|1999-06-13   |CAMPANIA      |ACERRA               |9669        |9669           |0     |
|1999-06-13   |CAMPANIA      |MONDRAGONE           |6468        |6468           |0     |
|1999-06-13   |VENETO        |VITTORIO VENETO      |7701        |7701           |0     |
|1999-06-13   |EMILIA ROMAGNA|FIORANO MODENESE     |5978        |5978           |0     |
|1999-06-13   |CAMPANIA      |FRATTAMAGGIORE       |11219       |11219          |0     |
|1999-06-13   |EMILIA ROMAGNA|SCANDIANO            |8704        |8704           |0     |
|1999-06-13   |PIEMON

---

## 6. Comparing DataFrames and RDDs

The Gold layer was built and queried with the DataFrame / Spark SQL API, which is
the right choice for tabular analytics: the Catalyst optimiser rewrites and
optimises the query plan, column pruning and predicate push-down happen
automatically, and the syntax is familiar to anyone who knows SQL.

RDDs sit one level lower. They give you explicit control over every
transformation — `map`, `flatMap`, `filter`, `reduceByKey`, `sortBy` — at the
cost of writing more code and losing Catalyst optimisation.  
Below we re-implement **two of the analytical queries** from Section 5 using
the RDD API and compare the results.

| | DataFrame / SQL | RDD |
|---|---|---|
| Optimisation | Catalyst (automatic) | None (manual) |
| Schema awareness | Yes — typed columns | No — plain Python objects |
| Code verbosity | Low | High |
| Flexibility | Structured data | Any Python object |
| Best for | Tabular analytics | Custom aggregations, graph data, ML features |

### Setup — load Gold tables as DataFrames and expose their RDDs

In [65]:
import os
from pyspark.sql import functions as F

# Re-load from Gold Parquet (section is independently re-runnable)
GOLD_PATH = os.path.join(
    "/Users/hajdumarcell/Documents/GitHub/BDA_project",
    "/Users/hajdumarcell/Documents/GitHub/BDA_project",
    "Visualization", "data_parquet", "gold",
)

df_election  = spark.read.parquet(os.path.join(GOLD_PATH, "dim_election"))
df_geo       = spark.read.parquet(os.path.join(GOLD_PATH, "dim_geography"))
df_candidate = spark.read.parquet(os.path.join(GOLD_PATH, "dim_candidate"))
df_turnout   = spark.read.parquet(os.path.join(GOLD_PATH, "fact_turnout"))
df_cand_v    = spark.read.parquet(os.path.join(GOLD_PATH, "fact_candidate_votes"))

# Register views for the SQL side
for name, df in [
    ("dim_election", df_election),
    ("dim_geography", df_geo),
    ("dim_candidate", df_candidate),
    ("fact_turnout", df_turnout),
    ("fact_candidate_votes", df_cand_v),
]:
    df.createOrReplaceTempView(name)

print("Gold tables loaded and registered as SQL views.")

Gold tables loaded and registered as SQL views.


### Query 1 — National Turnout Rate by Year and Election Type

**DataFrame version** (identical to Section 5, Query 1 — shown here for direct comparison).

In [66]:
import time

t0 = time.perf_counter()
df_result_q1 = spark.sql("""
    SELECT
        e.year,
        e.election_type,
        SUM(t.turnout_total)      AS total_voters,
        SUM(t.registered_voters)  AS total_registered,
        ROUND(
            100.0 * SUM(t.turnout_total) / NULLIF(SUM(t.registered_voters), 0),
        2) AS turnout_pct
    FROM fact_turnout   t
    JOIN dim_election   e USING (election_id)
    JOIN dim_geography  g USING (geo_id)
    WHERE t.registered_voters > 0
      AND g.geo_scope != 'COUNTRY'
    GROUP BY e.year, e.election_type
    ORDER BY e.year, e.election_type
""")
df_result_q1.show(40)
time_df_q1 = time.perf_counter() - t0
print(f"DataFrame time: {time_df_q1:.2f}s")

+----+-------------+------------+----------------+-----------+
|year|election_type|total_voters|total_registered|turnout_pct|
+----+-------------+------------+----------------+-----------+
|1948|      CHAMBER|    26774306|        29033012|      92.22|
|1953|      CHAMBER|    28409140|        30276195|      93.83|
|1958|      CHAMBER|    30434270|        32434429|      93.83|
|1963|      CHAMBER|    31766009|        34199184|      92.89|
|1968|      CHAMBER|    33001644|        35566493|      92.79|
|1970|    MUNICIPAL|    26096389|        28419385|      91.83|
|1972|      CHAMBER|    34525661|        37049353|      93.19|
|1975|    MUNICIPAL|    27630326|        29942443|      92.28|
|1976|      CHAMBER|    37743785|        40447569|      93.32|
|1979|      CHAMBER|    38257912|        42213292|      90.63|
|1980|    MUNICIPAL|    28346980|        31732676|      89.33|
|1983|      CHAMBER|    39188182|        44526307|      88.01|
|1985|    MUNICIPAL|    31708787|        35348336|     

**RDD version** — same logic, expressed with low-level RDD primitives.

Steps:
1. Broadcast the two small dimension tables (election, geography) as Python dicts
   so each executor can do O(1) lookups without a shuffle join.
2. `filter` domestic, non-null turnout rows.
3. `map` each row to `((year, election_type), (turnout, registered))`.
4. `reduceByKey` sums the two-element tuples.
5. `map` computes the percentage and formats the output.
6. `sortBy` orders by year then election type.

In [67]:
# ── Step 1: broadcast dimension lookups as plain Python dicts ─────────────
t0 = time.perf_counter()

election_map = {
    row["election_id"]: (row["year"], row["election_type"])
    for row in df_election.collect()
}
domestic_geo_ids = {
    row["geo_id"]
    for row in df_geo.filter(F.col("geo_scope") != "COUNTRY").select("geo_id").collect()
}

bc_election_map  = spark.sparkContext.broadcast(election_map)
bc_domestic_geos = spark.sparkContext.broadcast(domestic_geo_ids)

# ── Step 2-4: RDD pipeline ────────────────────────────────────────────────
rdd_result_q1 = (
    df_turnout.rdd
    .filter(lambda r:
        r["geo_id"] in bc_domestic_geos.value
        and r["registered_voters"] is not None
        and r["registered_voters"] > 0
    )
    .map(lambda r: (
        bc_election_map.value.get(r["election_id"], (None, None)),
        (r["turnout_total"] or 0, r["registered_voters"] or 0),
    ))
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
    .map(lambda kv: (
        kv[0][0], kv[0][1], kv[1][0], kv[1][1],
        round(100.0 * kv[1][0] / kv[1][1], 2) if kv[1][1] else None,
    ))
    .sortBy(lambda r: (r[0] or 0, r[1] or ""))
)

print(f"{'year':<6} {'election_type':<12} {'total_voters':>14} {'total_registered':>18} {'turnout_pct':>12}")
print("-" * 66)
for row in rdd_result_q1.collect():
    year, etype, voters, registered, pct = row
    print(f"{str(year):<6} {str(etype):<12} {voters:>14,} {registered:>18,} {str(pct):>12}")

time_rdd_q1 = time.perf_counter() - t0
print(f"\nRDD time: {time_rdd_q1:.2f}s")

year   election_type   total_voters   total_registered  turnout_pct
------------------------------------------------------------------
1948   CHAMBER          26,774,306         29,033,012        92.22
1953   CHAMBER          28,409,140         30,276,195        93.83
1958   CHAMBER          30,434,270         32,434,429        93.83
1963   CHAMBER          31,766,009         34,199,184        92.89
1968   CHAMBER          33,001,644         35,566,493        92.79
1970   MUNICIPAL        26,096,389         28,419,385        91.83
1972   CHAMBER          34,525,661         37,049,353        93.19
1975   MUNICIPAL        27,630,326         29,942,443        92.28
1976   CHAMBER          37,743,785         40,447,569        93.32
1979   CHAMBER          38,257,912         42,213,292        90.63
1980   MUNICIPAL        28,346,980         31,732,676        89.33
1983   CHAMBER          39,188,182         44,526,307        88.01
1985   MUNICIPAL        31,708,787         35,348,336        

The two results are identical.

**Timing comparison — Query 1:**



---

### Query 5 — Gender Representation Among Elected Officials

**DataFrame version** (identical to Section 5, Query 5).

In [68]:
t0 = time.perf_counter()
df_result_q5 = spark.sql("""
    SELECT
        e.year,
        e.election_type,
        SUM(CASE WHEN c.sex = 'F' AND cv.elected THEN 1 ELSE 0 END) AS elected_female,
        SUM(CASE WHEN c.sex = 'M' AND cv.elected THEN 1 ELSE 0 END) AS elected_male,
        COUNT(CASE WHEN cv.elected THEN 1 END)                       AS total_elected,
        ROUND(
            100.0
            * SUM(CASE WHEN c.sex = 'F' AND cv.elected THEN 1 ELSE 0 END)
            / NULLIF(COUNT(CASE WHEN cv.elected THEN 1 END), 0),
        2) AS female_pct
    FROM fact_candidate_votes  cv
    JOIN dim_election           e  USING (election_id)
    JOIN dim_candidate          c  USING (candidate_id)
    WHERE c.sex IS NOT NULL
    GROUP BY e.year, e.election_type
    ORDER BY e.year, e.election_type
""")
df_result_q5.show(40)
time_df_q5 = time.perf_counter() - t0
print(f"DataFrame time: {time_df_q5:.2f}s")

+----+-------------+--------------+------------+-------------+----------+
|year|election_type|elected_female|elected_male|total_elected|female_pct|
+----+-------------+--------------+------------+-------------+----------+
|1948|      CHAMBER|             0|           0|            0|      NULL|
|1953|      CHAMBER|             0|           0|            0|      NULL|
|1958|      CHAMBER|             0|           0|            0|      NULL|
|1963|      CHAMBER|             0|           0|            0|      NULL|
|1968|      CHAMBER|             0|           0|            0|      NULL|
|1972|      CHAMBER|             0|           0|            0|      NULL|
|1976|      CHAMBER|             0|           0|            0|      NULL|
|1979|      CHAMBER|             0|           0|            0|      NULL|
|1983|      CHAMBER|             0|           0|            0|      NULL|
|1987|      CHAMBER|             0|           0|            0|      NULL|
|1992|      CHAMBER|             0|   

**RDD version** — conditional counting with tuple accumulators.

Each row maps to `((year, election_type), (female_elected, male_elected, total_elected))`
where each counter is 0 or 1. `reduceByKey` sums the three-element tuples in parallel.

In [69]:
t0 = time.perf_counter()

candidate_sex_map = {
    row["candidate_id"]: row["sex"]
    for row in df_candidate.filter(F.col("sex").isNotNull()).select("candidate_id", "sex").collect()
}
bc_candidate_sex = spark.sparkContext.broadcast(candidate_sex_map)

rdd_result_q5 = (
    df_cand_v.rdd
    .filter(lambda r: r["candidate_id"] in bc_candidate_sex.value)
    .map(lambda r: (
        bc_election_map.value.get(r["election_id"], (None, None)),
        (
            1 if (bc_candidate_sex.value[r["candidate_id"]] == "F" and r["elected"]) else 0,
            1 if (bc_candidate_sex.value[r["candidate_id"]] == "M" and r["elected"]) else 0,
            1 if r["elected"] else 0,
        )
    ))
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1], a[2] + b[2]))
    .map(lambda kv: (
        kv[0][0], kv[0][1],
        kv[1][0], kv[1][1], kv[1][2],
        round(100.0 * kv[1][0] / kv[1][2], 2) if kv[1][2] else None,
    ))
    .sortBy(lambda r: (r[0] or 0, r[1] or ""))
)

print(f"{'year':<6} {'election_type':<12} {'f_elected':>10} {'m_elected':>10} {'total':>8} {'female_pct':>11}")
print("-" * 61)
for row in rdd_result_q5.collect():
    year, etype, f_el, m_el, total, pct = row
    print(f"{str(year):<6} {str(etype):<12} {f_el:>10,} {m_el:>10,} {total:>8,} {str(pct):>11}")

time_rdd_q5 = time.perf_counter() - t0
print(f"\nRDD time: {time_rdd_q5:.2f}s")

year   election_type  f_elected  m_elected    total  female_pct
-------------------------------------------------------------
1948   CHAMBER               0          0        0        None
1953   CHAMBER               0          0        0        None
1958   CHAMBER               0          0        0        None
1963   CHAMBER               0          0        0        None
1968   CHAMBER               0          0        0        None
1972   CHAMBER               0          0        0        None
1976   CHAMBER               0          0        0        None
1979   CHAMBER               0          0        0        None
1983   CHAMBER               0          0        0        None
1987   CHAMBER               0          0        0        None
1992   CHAMBER               0          0        0        None
1993   MUNICIPAL             4        171      175        2.29
1994   CHAMBER               0          0        0        None
1994   MUNICIPAL             0         86       86     

#### Timing summary

```
Results are printed by the code cell below.
```

In [70]:
print("Query 1 — National Turnout by Year")
print(f"  DataFrame / SQL : {time_df_q1:.2f}s")
print(f"  RDD             : {time_rdd_q1:.2f}s")
print(f"  Speedup         : {time_rdd_q1 / time_df_q1:.1f}x slower" if time_rdd_q1 > time_df_q1 else f"  Speedup         : {time_df_q1 / time_rdd_q1:.1f}x faster")
print()
print("Query 5 — Gender Representation")
print(f"  DataFrame / SQL : {time_df_q5:.2f}s")
print(f"  RDD             : {time_rdd_q5:.2f}s")
print(f"  Speedup         : {time_rdd_q5 / time_df_q5:.1f}x slower" if time_rdd_q5 > time_df_q5 else f"  Speedup         : {time_df_q5 / time_rdd_q5:.1f}x faster")
print()
print("Note: times include Parquet read + execution. On a single local machine")
print("DataFrame overhead from Catalyst planning is visible at this data scale.")
print("At cluster scale the gap typically reverses in DataFrames' favour.")

Query 1 — National Turnout by Year
  DataFrame / SQL : 0.50s
  RDD             : 1.68s
  Speedup         : 3.4x slower

Query 5 — Gender Representation
  DataFrame / SQL : 0.53s
  RDD             : 2.68s
  Speedup         : 5.1x slower

Note: times include Parquet read + execution. On a single local machine
DataFrame overhead from Catalyst planning is visible at this data scale.
At cluster scale the gap typically reverses in DataFrames' favour.


#### When to use each API

| Scenario | Prefer |
|---|---|
| Tabular joins, aggregations, filtering | **DataFrame / SQL** — Catalyst optimises automatically |
| Custom Python objects (graphs, ML features, nested structures) | **RDD** — full flexibility |
| Explicit control over join strategy or shuffle | **RDD** — broadcast, `partitionBy`, combiner all visible in code |
| Interop with non-tabular libraries (NetworkX, scikit-learn) | **RDD** — `.map()` accepts any Python function |

In this project DataFrames are the right default. The RDD implementations above
produce identical results but require the developer to manage join strategy
(broadcast vs shuffle) and serialisation explicitly — work Catalyst handles
automatically. The timing numbers above show the practical cost of that extra work.